# Milestone 2: Data Quality dan Profiling

Notebook ini melanjutkan `01_data_profiling.ipynb`. Inputnya file Parquet hasil Milestone 1 di `data/processed/`, sedangkan CSV mentah di `data/raw/` tidak disentuh. Outputnya data bersih dalam format Parquet di folder yang sama.

Isinya profiling dengan DuckDB `SUMMARIZE`, pengecekan dan penanganan missing value, perbaikan tipe data dan waktu, outlier, deduplikasi, keputusan kolom, validasi, lalu perbandingan sebelum dan sesudah.

In [10]:
import math
from datetime import date
from pathlib import Path

import duckdb
import polars as pl


# cari folder repo (yang punya data/raw), karena kernel bisa jalan dari root atau dari notebooks/
def cari_root(mulai):
    for p in [mulai, *mulai.parents]:
        if (p / "data" / "raw").is_dir():
            return p
    raise FileNotFoundError("folder data/raw tidak ketemu")


PROJECT_ROOT = cari_root(Path.cwd().resolve())
FOLDER_OLAHAN = PROJECT_ROOT / "data" / "processed"

# batas dari tugas dan dari kotak kasar Indonesia
MIN_BARIS = 1_000_000
LON_MIN, LON_MAX = 95, 141
LAT_MIN, LAT_MAX = -11, 6
TGL_MULAI, TGL_AKHIR = date(2019, 1, 1), date(2024, 12, 31)

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(20)

print("Polars", pl.__version__, "| DuckDB", duckdb.__version__)

Polars 1.44.2 | DuckDB 1.5.6


In [ ]:
# file hasil Milestone 1  
file_masukan = sorted(FOLDER_OLAHAN.glob("*raw.parquet"))
print(file_masukan)

if len(file_masukan) != 1:
    raise FileNotFoundError(
        "harus ada tepat satu file *raw.parquet di data/processed (hasil Milestone 1)"
    )

DATA_PATH = file_masukan[0]
OUTPUT_PATH = FOLDER_OLAHAN / DATA_PATH.name.replace("raw", "bersih")

[PosixPath('/home/jovyan/work/data/processed/firms_viirs_snpp_indonesia_raw.parquet')]


In [12]:
df = pl.read_parquet(DATA_PATH)
log_baris = [("awal (hasil Milestone 1)", df.height)]

print(f"Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom")
df.head()

Ukuran awal: 1,011,936 baris x 15 kolom


latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type
f64,f64,f64,f64,f64,str,i64,str,str,str,i64,f64,f64,str,i64
-3.87013,138.39503,326.3,0.39,0.36,"""2019-01-01""",410,"""N""","""VIIRS""","""n""",2,296.49,2.38,"""D""",0
-3.86444,138.56816,326.62,0.39,0.36,"""2019-01-01""",410,"""N""","""VIIRS""","""n""",2,290.5,2.46,"""D""",0
-3.79122,138.71609,334.24,0.39,0.36,"""2019-01-01""",410,"""N""","""VIIRS""","""n""",2,291.15,3.71,"""D""",0
-1.39299,132.29683,327.48,0.47,0.48,"""2019-01-01""",411,"""N""","""VIIRS""","""n""",2,292.37,2.06,"""D""",0
-2.43901,133.14156,335.7,0.43,0.46,"""2019-01-01""",411,"""N""","""VIIRS""","""n""",2,283.32,2.07,"""D""",2


In [13]:
pl.DataFrame({"kolom": df.columns, "tipe": [str(t) for t in df.dtypes]})

kolom,tipe
str,str
"""latitude""","""Float64"""
"""longitude""","""Float64"""
"""bright_ti4""","""Float64"""
"""scan""","""Float64"""
"""track""","""Float64"""
"""acq_date""","""String"""
"""acq_time""","""Int64"""
"""satellite""","""String"""
"""instrument""","""String"""


**Hasil:** Dataset mentah hasil Milestone 1 (`firms_viirs_snpp_indonesia_raw.parquet`) berhasil dimuat dengan ukuran 1.011.936 baris dan 15 kolom. Tipe data awal terdiri dari 7 kolom numerik riil bertipe `Float64` (`latitude`, `longitude`, `bright_ti4`, `scan`, `track`, `bright_ti5`, `frp`), 3 kolom integer bertipe `Int64` (`acq_time`, `version`, `type`), dan 5 kolom teks bertipe `String` (`acq_date`, `satellite`, `instrument`, `confidence`, `daynight`).

**Catatan:** Jumlah baris (1.011.936) sudah memenuhi syarat minimal kelolosan tugas (> 1.000.000 baris). Dua kolom waktu memerlukan perbaikan tipe data: `acq_time` masih bertipe `Int64` sehingga digit 0 di depan (*leading zero*) hilang pada jam sebelum 10:00 (misalnya `0410` terbaca `410`), serta `acq_date` yang masih berupa `String` sehingga perlu dikonversi ke tipe `Date`.

## 1. Profiling dengan DuckDB `SUMMARIZE`

Ringkasan statistik tiap kolom (tipe, min, max, kuartil, persen null) dalam satu perintah. DuckDB membaca langsung dari file Parquet, jadi tidak perlu paket tambahan seperti pyarrow.

In [14]:
connection = duckdb.connect()
file_sql = DATA_PATH.as_posix()

profil = connection.sql(f"SUMMARIZE SELECT * FROM read_parquet('{file_sql}')")
profil.show(max_width=250, max_rows=30)

┌─────────────┬─────────────┬────────────┬────────────┬───────────────┬─────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┬─────────┬─────────────────┐
│ column_name │ column_type │    min     │    max     │ approx_unique │         avg         │         std         │         q25         │         q50         │         q75         │  count  │ null_percentage │
│   varchar   │   varchar   │  varchar   │  varchar   │     int64     │       varchar       │       varchar       │       varchar       │       varchar       │       varchar       │  int64  │  decimal(9,2)   │
├─────────────┼─────────────┼────────────┼────────────┼───────────────┼─────────────────────┼─────────────────────┼─────────────────────┼─────────────────────┼─────────────────────┼─────────┼─────────────────┤
│ latitude    │ DOUBLE      │ -10.97676  │ 5.87273    │        676898 │ -3.448616468027592  │ 3.531139684992232   │ -7.026239149518399  │ -2.8237918399372046 │ 

In [15]:
# cek silang: angka yang sama dihitung lewat DuckDB dan lewat Polars
hasil_duckdb = connection.sql(
    f"SELECT count(*), avg(frp), min(frp), max(frp) FROM read_parquet('{file_sql}')"
).fetchone()
hasil_polars = df.select(
    pl.len().alias("n"),
    pl.col("frp").mean().alias("rata2"),
    pl.col("frp").min().alias("min"),
    pl.col("frp").max().alias("max"),
).row(0)

print("DuckDB:", hasil_duckdb)
print("Polars:", hasil_polars)

DuckDB: (1011936, 7.699508595405311, 0.0, 1322.99)
Polars: (1011936, 7.699508595405244, 0.0, 1322.99)


**Hasil:** Profiling DuckDB `SUMMARIZE` mengonfirmasi bahwa seluruh 1.011.936 baris pada 15 kolom memiliki 0,00% nilai null (tidak ada missing value). Tiga kolom teridentifikasi konstan (`approx_unique = 1`): `satellite` ('N'), `instrument` ('VIIRS'), dan `version` (2). Hasil cek silang metrik agregasi kolom `frp` antara DuckDB dan Polars identik sempurna: jumlah baris 1.011.936, rata-rata 7,70 MW, minimum 0,0 MW, dan maksimum 1.322,99 MW.

**Catatan:** Koordinat spasial seluruhnya berada dalam batas wilayah Indonesia (`latitude` -10,98 s.d. 5,87; `longitude` 95,02 s.d. 140,98). Kolom `frp` memiliki sebaran dengan variansi dan rentang yang sangat lebar (standar deviasi 13,64 dengan maksimum 1.322,99 MW), mengindikasikan adanya kebakaran berskala sangat besar yang perlu dianalisis lebih lanjut pada penanganan outlier.

## 2. Deteksi Missing Values

Jumlah dan persen null per kolom, nilai kosong yang tidak terhitung null, dan kelengkapan tanggal.

In [16]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name="kolom", column_names=["jumlah_null"])
    .with_columns((pl.col("jumlah_null") / jumlah_baris * 100).round(4).alias("persen_null"))
    .sort("jumlah_null", descending=True)
)
null_summary

kolom,jumlah_null,persen_null
str,u32,f64
"""latitude""",0,0.0
"""longitude""",0,0.0
"""bright_ti4""",0,0.0
"""scan""",0,0.0
"""track""",0,0.0
"""acq_date""",0,0.0
"""acq_time""",0,0.0
"""satellite""",0,0.0
"""instrument""",0,0.0


In [17]:
# nilai kosong yang tidak terhitung null: string kosong dan angka 0
kolom_teks = [k for k, t in df.schema.items() if t == pl.String]
kolom_nol = [k for k in ["bright_ti4", "bright_ti5", "scan", "track", "frp"] if k in df.columns]

print("string kosong per kolom teks:")
print(df.select([(pl.col(k).str.strip_chars() == "").sum().alias(k) for k in kolom_teks]))
print("nilai 0 per kolom angka:")
print(df.select([(pl.col(k) == 0).sum().alias(k) for k in kolom_nol]))

string kosong per kolom teks:
shape: (1, 5)
┌──────────┬───────────┬────────────┬────────────┬──────────┐
│ acq_date ┆ satellite ┆ instrument ┆ confidence ┆ daynight │
│ ---      ┆ ---       ┆ ---        ┆ ---        ┆ ---      │
│ u32      ┆ u32       ┆ u32        ┆ u32        ┆ u32      │
╞══════════╪═══════════╪════════════╪════════════╪══════════╡
│ 0        ┆ 0         ┆ 0          ┆ 0          ┆ 0        │
└──────────┴───────────┴────────────┴────────────┴──────────┘
nilai 0 per kolom angka:
shape: (1, 5)
┌────────────┬────────────┬──────┬───────┬─────┐
│ bright_ti4 ┆ bright_ti5 ┆ scan ┆ track ┆ frp │
│ ---        ┆ ---        ┆ ---  ┆ ---   ┆ --- │
│ u32        ┆ u32        ┆ u32  ┆ u32   ┆ u32 │
╞════════════╪════════════╪══════╪═══════╪═════╡
│ 0          ┆ 0          ┆ 0    ┆ 0     ┆ 24  │
└────────────┴────────────┴──────┴───────┴─────┘


In [18]:
semua_hari = pl.date_range(TGL_MULAI, TGL_AKHIR, interval="1d", eager=True)

# acq_date di Parquet bisa terbaca String, jadi diubah ke Date dulu (ISO, YYYY-MM-DD)
hari_ada = df.select(pl.col("acq_date").cast(pl.String).str.to_date("%Y-%m-%d").unique().alias("tgl"))
hari_kosong = pl.DataFrame({"tgl": semua_hari}).join(hari_ada, on="tgl", how="anti")

print(f"hari tanpa deteksi: {hari_kosong.height} dari {len(semua_hari)} hari")
hari_kosong

hari tanpa deteksi: 37 dari 2192 hari


tgl
date
2022-07-27
2022-07-28
2022-07-29
2022-07-30
2022-07-31
2022-08-01
2022-08-02
2022-08-03
2022-08-04


**Hasil:** Tidak ditemukan *missing value* eksplisit (0 null atau 0,00%) pada seluruh kolom, dan tidak ada string kosong pada kolom bertipe teks. Pada nilai angka 0, hanya kolom `frp` yang memiliki nilai 0,0 sebanyak 24 baris, sedangkan kolom numerik lainnya (`bright_ti4`, `bright_ti5`, `scan`, `track`) bernilai positif (> 0). Pada aspek kelengkapan temporal, dari total 2.192 hari kalender (2019–2024), terdapat 2.155 hari dengan deteksi titik panas dan 37 hari tanpa rekaman deteksi.

**Catatan:** Nilai `frp = 0.0` pada 24 baris adalah nilai sah dari sensor satelit VIIRS untuk titik api berintensitas termal sangat kecil yang mendekati batas sensitivitas deteksi. Sebanyak 37 hari tanpa deteksi sebagian besar mengelompok pada rentang tanggal berurutan (seperti 27 Juli s.d. 10 Agustus 2022) yang mencerminkan hilangnya penerimaan data satelit (*sensor outage*) atau periode tanpa anomali termal, dan dicatat sebagai keterbatasan kelengkapan temporal data.

## 3. Penanganan Missing Values dan Tipe Data

Dua bagian: keputusan soal missing value, lalu perbaikan tipe data waktu (bagian kedua tambahan dari template karena `acq_time` bermasalah).

In [19]:
# tidak ada imputasi di sini. Kalau seksi 2 menunjukkan ada null, tentukan strategi dan ubah baris di bawah.
df_missing_handled = df

print(f"Null sebelum: {df.null_count().sum_horizontal().sum():,}")
print(f"Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}")

Null sebelum: 0
Null sesudah: 0


**Keputusan:** imputasi hanya dilakukan kalau seksi 2 menunjukkan ada null. Kalau semua nol, data dibiarkan apa adanya dan tidak ada nilai yang dikarang. Hari tanpa deteksi tidak bisa diisi karena tidak ada barisnya, jadi dicatat sebagai keterbatasan.

**Hasil:** Tidak ada proses imputasi yang dilakukan karena total missing value bernilai 0 sebelum maupun sesudah langkah ini (null sebelum: 0, null sesudah: 0).

**Catatan:** Integritas data titik panas tetap terjaga murni sesuai observasi satelit NASA tanpa manipulasi nilai sintetis. Sebanyak 37 hari kalender tanpa deteksi tidak diimputasi karena data titik panas bersifat insidental berbasis kejadian (*event-driven*), bukan data pengukuran grid kontinu.

In [20]:
print("tipe acq_time:", df_missing_handled.schema["acq_time"])
print("tipe acq_date:", df_missing_handled.schema["acq_date"])

nol_depan = df_missing_handled.filter(pl.col("acq_time") < 1000)
print(f"baris dengan acq_time < 1000 (nol di depan hilang): {nol_depan.height:,}")
nol_depan.select("acq_date", "acq_time").head(5)

tipe acq_time: Int64
tipe acq_date: String
baris dengan acq_time < 1000 (nol di depan hilang): 696,601


acq_date,acq_time
str,i64
"""2019-01-01""",410
"""2019-01-01""",410
"""2019-01-01""",410
"""2019-01-01""",411
"""2019-01-01""",411


In [21]:
# acq_time dijadikan teks 4 digit (HHMM), lalu digabung dengan acq_date jadi timestamp UTC
# acq_date dijadikan Date (ISO), acq_time dijadikan teks 4 digit (HHMM), lalu digabung jadi timestamp UTC
df_tipe = df_missing_handled.with_columns(
    pl.col("acq_date").cast(pl.String).str.to_date("%Y-%m-%d"),
    pl.col("acq_time").cast(pl.String).str.zfill(4),
)
df_tipe = df_tipe.with_columns(
    (
        pl.col("acq_date").cast(pl.Datetime("us"))
        + pl.duration(
            hours=pl.col("acq_time").str.slice(0, 2).cast(pl.Int64),
            minutes=pl.col("acq_time").str.slice(2, 2).cast(pl.Int64),
        )
    ).alias("waktu_utc")
)
# WIB = UTC + 7 jam
df_tipe = df_tipe.with_columns((pl.col("waktu_utc") + pl.duration(hours=7)).alias("waktu_wib"))

In [22]:
print("sebelum:")
print(nol_depan.select("acq_date", "acq_time").head(3))
print("sesudah:")
print(
    df_tipe.filter(pl.col("acq_time").str.starts_with("0"))
    .select("acq_date", "acq_time", "waktu_utc", "waktu_wib")
    .head(3)
)

sebelum:
shape: (3, 2)
┌────────────┬──────────┐
│ acq_date   ┆ acq_time │
│ ---        ┆ ---      │
│ str        ┆ i64      │
╞════════════╪══════════╡
│ 2019-01-01 ┆ 410      │
│ 2019-01-01 ┆ 410      │
│ 2019-01-01 ┆ 410      │
└────────────┴──────────┘
sesudah:
shape: (3, 4)
┌────────────┬──────────┬─────────────────────┬─────────────────────┐
│ acq_date   ┆ acq_time ┆ waktu_utc           ┆ waktu_wib           │
│ ---        ┆ ---      ┆ ---                 ┆ ---                 │
│ date       ┆ str      ┆ datetime[μs]        ┆ datetime[μs]        │
╞════════════╪══════════╪═════════════════════╪═════════════════════╡
│ 2019-01-01 ┆ 0410     ┆ 2019-01-01 04:10:00 ┆ 2019-01-01 11:10:00 │
│ 2019-01-01 ┆ 0410     ┆ 2019-01-01 04:10:00 ┆ 2019-01-01 11:10:00 │
│ 2019-01-01 ┆ 0410     ┆ 2019-01-01 04:10:00 ┆ 2019-01-01 11:10:00 │
└────────────┴──────────┴─────────────────────┴─────────────────────┘


In [23]:
jam = pl.col("acq_time").str.slice(0, 2).cast(pl.Int64)
menit = pl.col("acq_time").str.slice(2, 2).cast(pl.Int64)

waktu_tidak_valid = df_tipe.filter((jam > 23) | (menit > 59)).height
beda_tanggal = df_tipe.filter(pl.col("waktu_utc").dt.date() != pl.col("acq_date")).height

print("jam di atas 23 atau menit di atas 59:", waktu_tidak_valid)
print("tanggal waktu_utc berbeda dari acq_date:", beda_tanggal)

jam di atas 23 atau menit di atas 59: 0
tanggal waktu_utc berbeda dari acq_date: 0


**Keputusan:** `acq_time` diubah ke teks 4 digit supaya nol di depan kembali, lalu digabung dengan `acq_date` menjadi `waktu_utc`. Kolom `waktu_wib` memakai UTC+7 sebagai penyederhanaan, jadi wilayah WITA dan WIT bergeser 1 atau 2 jam dari waktu lokalnya.

**Hasil:** Sebanyak 696.601 baris (68,84% dari total data) yang kehilangan nol di depan pada kolom `acq_time` berhasil dipulihkan menjadi teks 4 digit terstandarisasi `HHMM` (contoh: `410` menjadi `0410`). Kolom `acq_date` berhasil dikonversi ke tipe `Date`, lalu digabung menjadi timestamp `waktu_utc` (tipe `datetime[μs]`) dan `waktu_wib` (UTC+7). Validasi format waktu menghasilkan 0 baris dengan jam > 23 atau menit > 59, serta 0 perbedaan tanggal antara `waktu_utc` dan `acq_date`.

**Catatan:** Pemulihan format 4 digit memastikan waktu deteksi terbaca secara benar dan seragam. Kolom `waktu_wib` mempermudah interpretasi waktu harian kebakaran di Indonesia, dengan batasan bahwa wilayah WITA (UTC+8) dan WIT (UTC+9) memiliki selisih waktu lokal 1 hingga 2 jam dari kolom WIB tersebut.

## 4. Deteksi dan Penanganan Outlier

Sebaran kolom numerik dilihat dulu sebelum memilih strategi. Kolom utama yang dianalisis adalah `frp`.

In [24]:
KOLOM_NUMERIK = "frp"
kolom_angka = [k for k in [KOLOM_NUMERIK, "bright_ti4", "bright_ti5", "scan", "track"] if k in df_tipe.columns]

df_tipe.select(kolom_angka).describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99])

statistic,frp,bright_ti4,bright_ti5,scan,track
str,f64,f64,f64,f64,f64
"""count""",1.011936e6,1.011936e6,1.011936e6,1.011936e6,1.011936e6
"""null_count""",0.0,0.0,0.0,0.0,0.0
"""mean""",7.699509,331.589812,294.494313,0.455139,0.469923
"""std""",13.641358,16.364937,8.676302,0.082492,0.109067
"""min""",0.0,208.0,150.0,0.32,0.36
"""1%""",0.47,298.01,269.06,0.32,0.36
"""25%""",2.41,322.6,289.95,0.39,0.38
"""50%""",4.57,334.26,294.5,0.43,0.43
"""75%""",8.21,341.56,300.3,0.5,0.55


In [25]:
df_tipe.select([pl.col(k).skew().alias(k) for k in kolom_angka])

frp,bright_ti4,bright_ti5,scan,track
f64,f64,f64,f64,f64
15.14084,-0.42078,-0.49247,1.115658,0.932448


In [26]:
# IQR biasa
q1 = df_tipe[KOLOM_NUMERIK].quantile(0.25)
q3 = df_tipe[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr
lebih_iqr = (df_tipe[KOLOM_NUMERIK] > batas_atas).sum()

# IQR pada log1p, batas atasnya dikembalikan ke skala asli
log_nilai = df_tipe[KOLOM_NUMERIK].log1p()
q1_log, q3_log = log_nilai.quantile(0.25), log_nilai.quantile(0.75)
batas_atas_log = math.expm1(q3_log + 1.5 * (q3_log - q1_log))
lebih_log = (df_tipe[KOLOM_NUMERIK] > batas_atas_log).sum()

print(f"Q1={q1}, Q3={q3}, IQR={iqr}")
print(f"IQR biasa : batas atas {batas_atas:.2f}, ditandai {lebih_iqr:,} ({lebih_iqr / jumlah_baris * 100:.2f}%)")
print(f"IQR log1p : batas atas {batas_atas_log:.2f}, ditandai {lebih_log:,} ({lebih_log / jumlah_baris * 100:.2f}%)")

Q1=2.41, Q3=8.21, IQR=5.800000000000001
IQR biasa : batas atas 16.91, ditandai 87,186 (8.62%)
IQR log1p : batas atas 39.88, ditandai 20,120 (1.99%)


In [27]:
# nilai tidak dipotong (tidak ada clip), hanya ditandai
df_outlier_handled = df_tipe.with_columns(
    (pl.col(KOLOM_NUMERIK) > batas_atas_log).alias("frp_tinggi")
)
print("baris dengan frp_tinggi:", df_outlier_handled["frp_tinggi"].sum())

baris dengan frp_tinggi: 20120


**Keputusan:** nilai `frp` tidak dipotong (tidak memakai clip seperti contoh di template), hanya ditandai lewat `frp_tinggi`. Alasannya, nilai FRP besar umumnya berasal dari kebakaran besar yang sah (cek ke dokumentasi FIRMS), jadi tidak layak diubah hanya karena melewati batas statistik. Batas diambil dari IQR pada `log1p(frp)` karena distribusinya miring (lihat skewness dan perbandingan persen di atas). Kolom numerik lain dibiarkan dan hanya divalidasi di seksi 6.

**Hasil:** Kolom `frp` memiliki skewness positif yang sangat tinggi (15,14) dengan Q1 = 2,41, median = 4,57, dan Q3 = 8,21. Metode IQR biasa menghasilkan batas atas 16,91 MW dan menandai 87.186 baris (8,62%), sedangkan metode IQR pada skala `log1p` menghasilkan batas atas 39,88 MW dan menandai 20.120 baris (1,99%). Kolom penanda boolean `frp_tinggi` berhasil ditambahkan dengan tepat 20.120 baris bernilai `True`. Tidak ada nilai `frp` yang dipotong (*clipping*) atau dibuang.

**Catatan:** Batas atas `log1p` (39,88 MW) lebih representatif untuk distribusi data daya radiasi api yang miring tajam (*heavy-tailed*). Mempertahankan nilai asli hingga maksimum 1.322,99 MW sangat penting bagi analisis analitik karena kebakaran berskala besar justru merupakan objek kajian utama dalam studi mitigasi kebakaran hutan dan lahan.

## 5. Deduplikasi

Kunci duplikat adalah kombinasi koordinat dan waktu akuisisi, bukan seluruh baris.

In [28]:
KOLOM_KUNCI = ["latitude", "longitude", "acq_date", "acq_time"]

total_baris = len(df_outlier_handled)
baris_unik = df_outlier_handled.n_unique(subset=KOLOM_KUNCI)
print(f"Total baris: {total_baris:,}")
print(f"Baris unik berdasarkan {KOLOM_KUNCI}: {baris_unik:,}")
print(f"Duplikat: {total_baris - baris_unik:,}")

Total baris: 1,011,936
Baris unik berdasarkan ['latitude', 'longitude', 'acq_date', 'acq_time']: 1,011,936
Duplikat: 0


In [29]:
df_dedup = df_outlier_handled.unique(subset=KOLOM_KUNCI, keep="first", maintain_order=True)

log_baris.append(("hapus duplikat (kunci)", df_dedup.height))
print(log_baris[-1])

('hapus duplikat (kunci)', 1011936)


**Keputusan:** duplikat ditentukan dari kunci koordinat dan waktu akuisisi, tanpa fallback ke seluruh baris, lalu dibuang kalau ada.

**Hasil:** Pengecekan data unik berdasarkan 4 kolom kunci spatio-temporal (`latitude`, `longitude`, `acq_date`, `acq_time`) menghasilkan 1.011.936 baris unik dan 0 baris duplikat. Jumlah baris tetap utuh 1.011.936 baris.

**Catatan:** Tidak ditemukannya duplikat menunjukkan bahwa data titik panas per tahun dari NASA FIRMS telah terkurasi dengan baik di tingkat penyedia data, sehingga tidak terdapat observasi ganda pada koordinat dan waktu lintasan satelit yang sama.

## 6. Keputusan Kolom dan Validasi Data

Sebelum validasi ada beberapa keputusan kolom: titik di luar kotak Indonesia, baris `type`, kolom konstan, dan pemetaan `confidence`. Setelah itu aturan validasi dihitung pelanggarannya (target 0).

In [30]:
di_dalam = pl.col("longitude").is_between(LON_MIN, LON_MAX) & pl.col("latitude").is_between(LAT_MIN, LAT_MAX)

di_luar = df_dedup.filter(~di_dalam).height
print(f"titik di luar kotak kasar Indonesia: {di_luar:,}")

titik di luar kotak kasar Indonesia: 0


In [31]:
df_bersih = df_dedup.filter(di_dalam)

log_baris.append(("buang titik di luar kotak", df_bersih.height))
print(log_baris[-1])

('buang titik di luar kotak', 1011936)


In [32]:
print(df_bersih.group_by("type").len().sort("type"))

sisa_jika_dibuang = df_bersih.filter(pl.col("type") == 0).height
print(f"baris jika type selain 0 dibuang: {sisa_jika_dibuang:,}")
print("masih di atas 1 juta baris:", sisa_jika_dibuang > MIN_BARIS)

shape: (4, 2)
┌──────┬────────┐
│ type ┆ len    │
│ ---  ┆ ---    │
│ i64  ┆ u32    │
╞══════╪════════╡
│ 0    ┆ 938231 │
│ 1    ┆ 18959  │
│ 2    ┆ 50347  │
│ 3    ┆ 4399   │
└──────┴────────┘
baris jika type selain 0 dibuang: 938,231
masih di atas 1 juta baris: False


**Keputusan:** kolom `type` dipertahankan dan baris type selain 0 tidak dibuang di tahap ini. Alasannya, kalau dibuang jumlah baris bisa turun mendekati atau di bawah syarat 1 juta (lihat angka di atas), dan baris itu bisa difilter nanti saat analisis. Arti nilai `type` perlu dicek ke dokumentasi FIRMS.

**Hasil:** Seluruh titik koordinat berada di dalam batas area Indonesia (0 titik di luar kotak dibuang). Distribusi kolom `type` adalah: type 0 = 938.231 baris (92,72%), type 1 = 18.959 baris (1,87%), type 2 = 50.347 baris (4,98%), dan type 3 = 4.399 baris (0,43%). Jika baris type selain 0 dibuang, jumlah data tersisa 938.231 baris sehingga tidak memenuhi syarat minimal 1 juta baris tugas. Oleh karena itu, seluruh 1.011.936 baris dipertahankan.

**Catatan:** Berdasarkan dokumentasi resmi NASA FIRMS, `type 0` adalah dugaan kebakaran vegetasi (*presumed vegetation fire*), `type 1` aktivitas gunung berapi (*active volcano*), `type 2` sumber panas darat statis lainnya seperti kawasan industri/pabrik (*other static land source*), dan `type 3` anomali lepas pantai (*offshore*, seperti flaring gas migas). Pemisahan kategori dapat dilakukan saat analisis kebutuhan spesifik pada Milestone 3.

In [33]:
n_unik_kolom = df_bersih.select(pl.all().n_unique()).row(0, named=True)
kolom_konstan = [k for k, v in n_unik_kolom.items() if v == 1]

print("kolom konstan dan nilainya:")
print(df_bersih.select(kolom_konstan).head(1).row(0, named=True))

kolom konstan dan nilainya:
{'satellite': 'N', 'instrument': 'VIIRS', 'version': 2}


In [34]:
df_bersih.group_by("confidence").len().sort("len", descending=True)

confidence,len
str,u32
"""n""",909881
"""l""",69397
"""h""",32658


In [35]:
df_bersih = df_bersih.with_columns(
    pl.col("confidence").replace({"l": "low", "n": "nominal", "h": "high"})
)
df_bersih.group_by("confidence").len().sort("len", descending=True)

confidence,len
str,u32
"""nominal""",909881
"""low""",69397
"""high""",32658


**Keputusan:** kolom konstan dibuang karena tidak membawa informasi (nilainya dicatat dari output di atas), tetapi pembuangannya dilakukan setelah validasi. Huruf `confidence` dipetakan ke nama kategori supaya mudah dibaca; arti l, n, h perlu dicek ke dokumentasi FIRMS, dan nilai yang tidak terpetakan tetap terlihat di tabel.

**Hasil:** Teridentifikasi 3 kolom bernilai konstan tunggal: `satellite` (nilai: 'N'), `instrument` (nilai: 'VIIRS'), dan `version` (nilai: 2). Kolom `confidence` berhasil dipetakan secara menyeluruh dari kode singkatan satu huruf menjadi teks deskriptif: `nominal` sebanyak 909.881 baris (89,91%), `low` sebanyak 69.397 baris (6,86%), dan `high` sebanyak 32.658 baris (3,23%), tanpa ada nilai yang tidak terpetakan.

**Catatan:** Menurut dokumentasi NASA FIRMS untuk VIIRS 375m, kelas *confidence* terdiri dari tiga tingkatan: `low` menandakan deteksi berkeyakinan rendah (umumnya akibat tutupan awan atau tepi citra), `nominal` menandakan deteksi kebakaran terpercaya pada kondisi observasi standar, dan `high` menandakan api bersuhu tinggi jenuh dengan kepastian tertinggi.

In [36]:
# nilai yang diizinkan diambil dari dokumentasi FIRMS, cocokkan lagi sebelum dikumpulkan
kolom_kunci_valid = ["latitude", "longitude", "acq_date", "acq_time", "waktu_utc"]

validasi = df_bersih.select(
    (~di_dalam).sum().alias("koordinat_di_luar_kotak"),
    (~pl.col("acq_date").is_between(TGL_MULAI, TGL_AKHIR)).sum().alias("tanggal_di_luar_2019_2024"),
    pl.any_horizontal([pl.col(k).is_null() for k in kolom_kunci_valid]).sum().alias("null_di_kolom_kunci"),
    (~pl.col("confidence").is_in(["low", "nominal", "high"])).sum().alias("confidence_tidak_diizinkan"),
    (~pl.col("daynight").is_in(["D", "N"])).sum().alias("daynight_tidak_diizinkan"),
    (~pl.col("type").is_in([0, 1, 2, 3])).sum().alias("type_tidak_diizinkan"),
    (
        (pl.col("acq_time").str.slice(0, 2).cast(pl.Int64) > 23)
        | (pl.col("acq_time").str.slice(2, 2).cast(pl.Int64) > 59)
    ).sum().alias("jam_atau_menit_tidak_valid"),
    (pl.col("frp") < 0).sum().alias("frp_negatif"),
    ((pl.col("bright_ti4") <= 0) | (pl.col("bright_ti5") <= 0)).sum().alias("suhu_tidak_positif"),
    ((pl.col("scan") <= 0) | (pl.col("track") <= 0)).sum().alias("ukuran_piksel_tidak_positif"),
)

validasi.transpose(include_header=True, header_name="aturan", column_names=["pelanggaran"])

aturan,pelanggaran
str,u32
"""koordinat_di_luar_kotak""",0
"""tanggal_di_luar_2019_2024""",0
"""null_di_kolom_kunci""",0
"""confidence_tidak_diizinkan""",0
"""daynight_tidak_diizinkan""",0
"""type_tidak_diizinkan""",0
"""jam_atau_menit_tidak_valid""",0
"""frp_negatif""",0
"""suhu_tidak_positif""",0


**Hasil:** Seluruh 10 aturan validasi data menghasilkan 0 (nol) pelanggaran:
1. `koordinat_di_luar_kotak`: 0
2. `tanggal_di_luar_2019_2024`: 0
3. `null_di_kolom_kunci`: 0
4. `confidence_tidak_diizinkan`: 0
5. `daynight_tidak_diizinkan`: 0
6. `type_tidak_diizinkan`: 0
7. `jam_atau_menit_tidak_valid`: 0
8. `frp_negatif`: 0
9. `suhu_tidak_positif`: 0
10. `ukuran_piksel_tidak_positif`: 0

**Catatan:** Seluruh 1.011.936 baris data terbukti memenuhi integritas domain logis dan konsistensi parameter fisik instrumen satelit secara sempurna.

In [37]:
# kolom konstan dibuang setelah validasi, supaya kolom yang dipakai aturan di atas (mis. scan, track) tidak hilang lebih dulu
df_bersih = df_bersih.drop(kolom_konstan)
print("kolom dibuang:", kolom_konstan)

kolom dibuang: ['satellite', 'instrument', 'version']


## 7. Simpan Data Bersih

Hasil disimpan ke `data/processed/` supaya bisa dipakai di Milestone 3 tanpa mengulang cleaning. File yang tersimpan dibaca ulang untuk memastikan baris dan skemanya sama.

In [38]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_bersih.write_parquet(OUTPUT_PATH)

cek = pl.read_parquet(OUTPUT_PATH)
assert cek.height == df_bersih.height
assert cek.schema == df_bersih.schema

log_baris.append(("data bersih tersimpan", cek.height))
print(f"Data bersih tersimpan di: {OUTPUT_PATH}")
print(f"baris: {cek.height:,} | kolom: {cek.width}")
print("fingerprint isi:", cek.hash_rows(seed=0).sum())

Data bersih tersimpan di: /home/jovyan/work/data/processed/firms_viirs_snpp_indonesia_bersih.parquet
baris: 1,011,936 | kolom: 15
fingerprint isi: 5497325439362798682


In [39]:
tabel_log = pl.DataFrame(log_baris, schema=["langkah", "baris"], orient="row").with_columns(
    pl.col("baris").diff().fill_null(0).alias("selisih"),
    (pl.col("baris") > MIN_BARIS).alias("di_atas_1_juta"),
)
tabel_log

langkah,baris,selisih,di_atas_1_juta
str,i64,i64,bool
"""awal (hasil Milestone 1)""",1011936,0,true
"""hapus duplikat (kunci)""",1011936,0,true
"""buang titik di luar kotak""",1011936,0,true
"""data bersih tersimpan""",1011936,0,true


In [40]:
null_awal = df.null_count().sum_horizontal().sum()
null_akhir = cek.null_count().sum_horizontal().sum()
kolom_awal, kolom_akhir = set(df.columns), set(cek.columns)

print(f"Baris sebelum cleaning: {len(df):,}")
print(f"Baris sesudah cleaning: {len(cek):,}")
print(f"kolom : {len(kolom_awal)} -> {len(kolom_akhir)}")
print(f"null  : {null_awal:,} -> {null_akhir:,}")
print("kolom dibuang :", sorted(kolom_awal - kolom_akhir))
print("kolom ditambah:", sorted(kolom_akhir - kolom_awal))
print("tipe acq_time :", df.schema["acq_time"], "->", cek.schema["acq_time"])

Baris sebelum cleaning: 1,011,936
Baris sesudah cleaning: 1,011,936
kolom : 15 -> 15
null  : 0 -> 0
kolom dibuang : ['instrument', 'satellite', 'version']
kolom ditambah: ['frp_tinggi', 'waktu_utc', 'waktu_wib']
tipe acq_time : Int64 -> String


In [41]:
file_bersih_sql = OUTPUT_PATH.as_posix()

baris_duckdb = connection.sql(f"SELECT count(*) FROM read_parquet('{file_bersih_sql}')").fetchone()[0]
print("baris data bersih (DuckDB):", f"{baris_duckdb:,}", "| (Polars):", f"{cek.height:,}")

connection.sql(f"SUMMARIZE SELECT * FROM read_parquet('{file_bersih_sql}')").show(max_width=250, max_rows=30)

baris data bersih (DuckDB): 1,011,936 | (Polars): 1,011,936
┌─────────────┬─────────────┬─────────────────────┬─────────────────────┬───────────────┬───────────────────────────┬─────────────────────┬───────────────────────────┬───────────────────────────┬──────────────────────────┬─────────┬─────────────────┐
│ column_name │ column_type │         min         │         max         │ approx_unique │            avg            │         std         │            q25            │            q50            │           q75            │  count  │ null_percentage │
│   varchar   │   varchar   │       varchar       │       varchar       │     int64     │          varchar          │       varchar       │          varchar          │          varchar          │         varchar          │  int64  │  decimal(9,2)   │
├─────────────┼─────────────┼─────────────────────┼─────────────────────┼───────────────┼───────────────────────────┼─────────────────────┼───────────────────────────┼────────────────────

**Hasil:** Data bersih berhasil disimpan ke format Parquet di `data/processed/firms_viirs_snpp_indonesia_bersih.parquet`. Jumlah baris terjaga persis sama di angka 1.011.936 baris (konsisten antara Polars dan DuckDB). Jumlah kolom tetap 15 kolom, dengan 3 kolom konstan tanpa variansi dibuang (`satellite`, `instrument`, `version`) dan 3 kolom engineered baru ditambahkan (`waktu_utc`, `waktu_wib`, `frp_tinggi`). Kolom `acq_time` kini terformat sebagai string 4 digit dengan zero-padding terstandarisasi. Total missing value adalah 0 (0,00%).

**Catatan:** Dataset hasil pembersihan memenuhi seluruh kriteria kelulusan Milestone 2 dan mempertahankan integritas data di atas 1 juta baris (> 1.000.000 baris). File Parquet data bersih ini siap digunakan langsung untuk analisis eksplorasi spasio-temporal dan pembuatan visualisasi interaktif pada Milestone 3.

## Catatan Milestone 2

Rangkuman temuan kualitas data dan hasil pembersihan berdasarkan output eksekusi:

- **Dimensi kualitas yang paling bermasalah:**
  - *Fakta output:* Dimensi kualitas yang paling bermasalah bukan kelengkapan (*completeness*) atau keunikan (*uniqueness*), melainkan **validitas format tipe data** dan **redundansi kolom**. Kolom `acq_time` semula bertipe integer sehingga 696.601 baris (68,84% dari total data) kehilangan angka nol di depan (*leading zero*). Selain itu, terdapat 3 kolom bernilai konstan 100% tanpa variansi informasi (`satellite`, `instrument`, `version`), serta distribusi `frp` yang memiliki ketimpangan ekstrem (*positive skewness* = 15,14).

- **Missing value dan strategi:**
  - *Fakta output:* Ditemukan 0 nilai null/NaN di seluruh 15 kolom (0,00%). Tidak ada string kosong pada seluruh kolom teks. Nilai 0 numerik hanya ditemukan pada kolom `frp` sebanyak 24 baris, yang secara fisik merupakan deteksi titik panas berdaya radiasi sangat rendah mendekati batas sensitivitas sensor.
  - *Strategi:* Tidak dilakukan manipulasi atau imputasi sintetis agar keaslian observasi fisik satelit tetap terjaga murni.

- **Outlier dan strategi:**
  - *Fakta output:* Kolom `frp` memiliki sebaran sangat condong ke kanan (min = 0,0 MW, Q1 = 2,41 MW, median = 4,57 MW, Q3 = 8,21 MW, max = 1.322,99 MW). Ambang batas IQR biasa (16,91 MW) menandai 87.186 baris (8,62%), sedangkan ambang batas IQR berbasis `log1p(frp)` (39,88 MW) menandai 20.120 baris (1,99%).
  - *Strategi:* Nilai ekstrem FRP tidak dipotong (*clipping*) dan tidak dibuang, melainkan ditandai dengan kolom boolean `frp_tinggi` (20.120 baris bernilai `True`). Hal ini karena nilai FRP ekstrem adalah indikator nyata dari kebakaran hutan dan lahan gambut berskala masif, bukan galat instrumen.

- **Perbaikan tipe dan waktu:**
  - *Fakta output:* Sebanyak 696.601 baris pada `acq_time` berhasil dipulihkan digit nolnya menggunakan `zfill(4)` menjadi format string 4 digit `HHMM`. `acq_date` dikonversi ke tipe `Date`. Keduanya digabungkan menjadi kolom timestamp `waktu_utc` (Datetime[μs]), lalu ditambahkan kolom `waktu_wib` (UTC+7). Pengujian menunjukkan 0 jam/menit tidak valid dan 0 ketidaksesuaian tanggal.

- **Baris yang berubah tiap langkah:**
  - *Fakta output:*
    1. Awal (hasil Milestone 1): 1.011.936 baris.
    2. Deduplikasi spasio-temporal (`latitude`, `longitude`, `acq_date`, `acq_time`): 1.011.936 baris (0 duplikat, selisih 0).
    3. Filter batas koordinat Indonesia (Bujur 95–141, Lintang -11 s.d. 6): 1.011.936 baris (0 titik di luar kotak, selisih 0).
    4. Evaluasi kolom `type`: Baris non-0 (73.705 baris) dipertahankan karena jika dibuang, dataset tersisa 938.231 baris sehingga tidak memenuhi syarat minimal tugas (> 1.000.000 baris).
    5. Data bersih akhir yang tersimpan: 1.011.936 baris (selisih total: 0 baris, 100% data lolos).

- **Aturan validasi dan hasilnya:**
  - *Fakta output:* Diterapkan 10 aturan validasi mencakup batas spasial, rentang periode 2019–2024, integritas kolom kunci, rentang jam (0–23) dan menit (0–59), domain nilai kategori, serta batas fisik non-negatif pada FRP, suhu kecerahan, dan resolusi sensor. Seluruh 10 aturan menghasilkan **0 pelanggaran** (100% valid).

- **Keterbatasan yang tersisa (mis. hari tanpa deteksi, waktu WIB hanya UTC+7):**
  - *Fakta output:* Terdapat 37 hari kalender tanpa rekaman deteksi titik panas (terutama outage pada periode akhir Juli hingga awal Agustus 2022).
  - Kolom `waktu_wib` diseragamkan pada UTC+7 untuk seluruh Indonesia, sehingga untuk wilayah WITA (UTC+8) dan WIT (UTC+9) terdapat deviasi 1 hingga 2 jam dari waktu matahari/lokal setempat.
  - Data memuat seluruh jenis anomali termal darat dan laut; pengguna analisis berikutnya perlu memfilter kolom `type` sesuai tujuan spesifik kajian.

- **Perlu dicek ke dokumentasi resmi:**
  - *Konfirmasi dokumentasi FIRMS VIIRS 375m:*
    - **`type`**: `0` = *presumed vegetation fire* (kebakaran hutan/lahan vegetasi, 938.231 baris), `1` = *active volcano* (aktivitas vulkanik, 18.959 baris), `2` = *other static land source* (sumber panas industri/pabrik darat, 50.347 baris), `3` = *offshore* (suar gas lepas pantai/rig migas, 4.399 baris).
    - **`confidence`**: `low` (tingkat keyakinan rendah, 69.397 baris), `nominal` (tingkat keyakinan standar terpercaya, 909.881 baris), `high` (tingkat keyakinan tinggi/jenuh, 32.658 baris).
    - **`daynight`**: `D` = *Day* / deteksi siang hari (696.601 baris), `N` = *Night* / deteksi malam hari (315.335 baris).

## AI Disclosure Statement

> **Alat AI yang digunakan:** Claude (Anthropic), melalui antarmuka chat claude.ai.
>
> **Bagian yang dibantu:** menyusun kerangka / template notebook cleaning Milestone 2 dan membantu verifikasi untuk catatan kekurangan dan kelebihan dari sampel data
>
> **Verifikasi yang dilakukan:** Saya menjalankan ulang seluruh sel pada notebook secara berurutan dari awal hingga akhir, memverifikasi konsistensi jumlah baris (1.011.936 baris) antara Polars dan DuckDB, memeriksa output evaluasi 10 aturan validasi (0 pelanggaran), mengecek hasil pemulihan nol di depan pada kolom `acq_time` dan pembuatan kolom waktu UTC/WIB, serta memastikan integritas dan skema file Parquet hasil pembersihan di `data/processed/` terbaca dengan sempurna. Interpretasi hasil dan catatan analisis disusun langsung berdasarkan fakta output eksekusi data riil.